# Understanding and Implementing Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks by`(Lewis et al., 2020).`

# 

## 1. Introduction and Motivation

Consider that you are building a question answer but then as you proceed all of sudden you hit the wall. You realise that traditional large language models like GPT-3 or BERT store all their knowledge in their parameters (the weights of the neural network) but they static and do not changes with the world. For example if you will asked your personal chatbot which had knowledge till November 2022 that **When did Lionel Messi win the world cup?** then it will answer that **Messi never won the world cup** or **Maximum Messi has reached closes to world cup is to the final in 2014.** It has no of the events that happened after that duration of time.<br>

Now this approach has several other problems also. For example:

1. **Hallucinations:** Models sometimes sound very knowledgeable and sophisticated but they are giving completely trash information about a certain topic. If you have no idea about that field then there is no way to verify those facts while models are confidently making up the facts.

2. **Knowledge Capacity:** Models have limited amoung of knowledge because we have to fit them into parameters but we can't fit billions of parameters which are needed to store knowledge of whole world and also such systems are difficult to scale.

Now by RAG we combine best of both worlds:
1. **Retriver:** helps us find relevant documents from a huge text corpus.
2. **Generator:** conditions on the query and retrieved documents to produce factual answers.

### 1.1 Problem Which RAG Attempts to Solve

Closed-book generators (like vanilla BART or GPT) must store all facts within parameters but they hallucinate on unseen or conflicting queries. Yet most real world tasks such as QA, fact checking, dialogue and summarization require grounding answers in external knowledge. RAG treats doucments as hidden variables that we conclude from observable and measurable variables through a mathematical model. For example consider a query $x$ and model is retrieving the documents $z_1$, $z_2 ... $, and then marginalizes over them while generating the final answer $y$.

## 2. Architecture of RAG

RAG has **three main components**:
```

Input Query (x)
     │
     ▼
┌─────────────────────────┐
│   RETRIEVER p_η(z|x)    │  ← Finds relevant documents
│(Dense Passage Retriev)  │
└──────────┬──────────────┘
           │
           ▼
     Top-K Documents (z)
           │
           ▼
┌─────────────────────────┐
│  GENERATOR p_θ(y|x,z)   │  ← Generates answer
│       (BART)            │
└──────────┬──────────────┘
           │
           ▼
      Answer (y)

```

### 2.1 Understanding the Retriever
DPR stands for Dense Passage Retrieval which is a neural retrieval system introduced by Facebook AI [Karpukhin et al., 2020](https://arxiv.org/abs/2004.04906). It’s deeply connected to how RAG retrieves documents.

DPR represents both queries and documents as continuous vectors using BERT like encoders and retrieves relevant documents by computing vector similarity (typically inner product) between a query vector and millions of document vectors stored in an index. DPR replaced traditional keyword-based search (like TF-IDF, BM25) with a neural, semantic search that works well for open-domain question answering. The retriever hastwo BERT encoders:

- $\text{Query encoder: } q(x) = f_q(x) \in \mathbb{R}^d$
- $\text{Document encoder: } d(z) = f_d(z) \in \mathbb{R}^d$ 


Our query enocder takes input $x$ and converts it into a dense vector representation $q(x)$. Then our document encoder has already been run on the entire knowledge base (e.g., all of Wikipedia, split into 100-word chunks 16). It creates a vector $d(z)$ for every single document $z$. Finally all these document vectors are stored in a massive Document Index18. The model uses **Maximum Inner Product Search (MIPS)** to instantly find the top-K document vectors $d(z)$ that are closest to our query vector $q(x)$

> In layman terms retriever turns our question into a vector and finds the K Wikipedia chunks whose vectors are the most similar.


### 2.2 Inside the Generator
The generator (BART or T5) models the probability of generating the sequence $y = (y_1, \ldots, y_N)$:<br>
$p_\theta(y \mid x, z)
= \prod_{i=1}^{N} p_\theta(y_i \mid x, z, y_{1:i-1})$

Here we are performing following steps:

* **Input:** concatenate `[x; z]` as the encoder input (query + passage).
* **Output:** autoregressive decoding of (y_i) one token at a time.
* **Training:** teacher forcing — the decoder conditions on the ground-truth prefix $y_{1:i-1}$.
* **Inference:** uses beam search or nucleus sampling for decoding.

To explain in short generator is pre-trained sequence-to-sequence model specifically mostly BART-large. The Generator takes both the original input $x$ and the text of a retrieved document $z$ and simply concatenates them and finally auto-regressively generates the final answer $y$.


## 3. The Mathematics of RAG

Let's first establish our notation clearly:

- $x$ : Input query (e.g., "What is the capital of France?")
- $y$ : Output sequence (e.g., "Paris")
- $z$ : Retrieved document
- $y_i$ : The $i$-th token in output sequence
- $N$ : Length of output sequence

RAG models the probability of generating output $y$ given input $x$ as:<br>
$p(y|x) = \sum_{z \in \mathcal{Z}} p(y, z|x)$

Where:
- $\mathcal{Z}$ is the set of all possible documents
- $p(y, z|x)$ is the joint probability of document $z$ and output $y$ given input $x$

By the chain rule of probability:<br>
$p(y, z|x) = p(z|x) \cdot p(y|x, z)$

Therefore $p(y|x) = \sum_{z \in \mathcal{Z}} p(z|x) \cdot p(y|x, z)$

**Interpretation:**
- $p(z|x)$ : **Retriever** - How relevant is document $z$ to query $x$?
- $p(y|x, z)$ : **Generator** - How likely is answer $y$ given query $x$ and document $z$?
- The sum marginalizes over all possible documents

### 3.1 The Top-K Approximation

**Problem:** Summing over ALL documents in Wikipedia (21 million) is computationally infeasible so

**Solution:** Approximate by summing over only the **top-K** most relevant documents:

$$p(y|x) \approx \sum_{z \in \text{top-}K(p(\cdot|x))} p(z|x) \cdot p(y|x, z)$$

Where:
- $\text{top-}K(p(\cdot|x))$ returns the $K$ documents with highest $p(z|x)$.
- Typically $K \in \{5, 10\}$.
- This is the **key approximation** in RAG.


 ## 4. The Interaction Between the Two Blocks
 
 Summing over all documents in Wikipedia which are in millions is computationally impossible therefore the authors approximate this sum by only using the top-K documents found by the retriever.<br>
 The paper proposes two different models for how to do this:

### 4.1 RAG-Sequence Model

RAG-Sequence uses the **same document** for generating the entire output sequence:

$p_{\text{RAG-Seq}}(y|x) \approx \sum_{z \in \text{top-}K(p(\cdot|x))} p_\eta(z|x) \cdot p_\theta(y|x, z)$

Now expanding the generator term:

$p_{\text{RAG-Seq}}(y|x) \approx \sum_{z \in \text{top-}K(p(\cdot|x))} p_\eta(z|x) \prod_{i=1}^{N} p_\theta(y_i|x, z, y_{1:i-1})$

#### Step-by-Step Breakdown

- **Step 1: Retrieve Documents**

For query $x$ retrieve $K$ documents with highest relevance:

$\{z_1, z_2, \ldots, z_K\} = \text{top-}K(p_\eta(\cdot|x))$

- **Step 2: Compute Generation Probability for Each Document**

For each document $z_k$, compute the probability of generating the full sequence:

$p_\theta(y|x, z_k) = \prod_{i=1}^{N} p_\theta(y_i|x, z_k, y_{1:i-1})$

**Step 3: Weight by Retrieval Probability**

Weight each generation probability by how relevant the document is:

$\text{weighted-prob}_k = p_\eta(z_k|x) \cdot p_\theta(y|x, z_k)$

- **Step 4: Marginalize (Sum) Over Documents**

$p_{\text{RAG-Seq}}(y|x) = \sum_{k=1}^{K} p_\eta(z_k|x) \cdot p_\theta(y|x, z_k)$

#### Example Calculation

**Given:**
- Query: "What is the capital of France?"
- Target: "Paris"
- K = 3 documents retrieved

**Step-by-step:**

1. **Retrieval probabilities:**
   $p_\eta(z_1|x) = 0.6, \quad p_\eta(z_2|x) = 0.3, \quad p_\eta(z_3|x) = 0.1$

2. **Generation probabilities:**
   $p_\theta(\text{"Paris"}|x, z_1) = 0.8$
   $p_\theta(\text{"Paris"}|x, z_2) = 0.5$
   $p_\theta(\text{"Paris"}|x, z_3) = 0.2$

3. **Joint probabilities:**
   $0.6 \times 0.8 = 0.48$
   $0.3 \times 0.5 = 0.15$
   $0.1 \times 0.2 = 0.02$

4. **Marginal probability:**
   $p(\text{"Paris"}|x) = 0.48 + 0.15 + 0.02 = 0.65$


### 4.2 RAG-Token Model

RAG-Token can use **different documents for each token**:

$p_{\text{RAG-Token}}(y|x) \approx \prod_{i=1}^{N} \sum_{z \in \text{top-}K(p(\cdot|x))} p_\eta(z|x) \cdot p_\theta(y_i|x, z, y_{1:i-1})$

In RAG token model the sum and product are **swapped** compared to RAG-Sequence

#### Step-by-Step Breakdown

**For each output position $i$:**

- **Step 1: Compute Token Probability for Each Document**

For document $z_k$, compute probability of generating token $y_i$:

$p_\theta(y_i|x, z_k, y_{1:i-1})$

- **Step 2: Weight by Retrieval Probability**

$\text{weighted-prob}_{k,i} = p_\eta(z_k|x) \cdot p_\theta(y_i|x, z_k, y_{1:i-1})$

- **Step 3: Marginalize Over Documents for This Token**

$p(y_i|x, y_{1:i-1}) = \sum_{k=1}^{K} p_\eta(z_k|x) \cdot p_\theta(y_i|x, z_k, y_{1:i-1})$

- **Step 4: Multiply Across All Tokens**

$p_{\text{RAG-Token}}(y|x) = \prod_{i=1}^{N} p(y_i|x, y_{1:i-1})$

#### Mathematical Intuition

- For **RAG-Sequence** pick one document and generate entire sequence with it.

$p(y|x) = \underbrace{\sum_{z}}_{\text{document}} \underbrace{\prod_{i}}_{\text{tokens}} p_\eta(z|x) \cdot p_\theta(y_i|x,z,y_{<i})$

- For **RAG-Token:** for each token marginalize over documents

$p(y|x) = \underbrace{\prod_{i}}_{\text{tokens}} \underbrace{\sum_{z}}_{\text{document}} p_\eta(z|x) \cdot p_\theta(y_i|x,z,y_{<i})$



## 5. Decoding the Answers

Generating an answer at test time (decoding) is tricky because of the marginalization.

- For RAG-Token this is simple. At each step $i$ the model calculates the next-token probability $p'(y_i | ...)$ by summing over the top-K documents. This final and blended probability distribution can be plugged directly into a standard beam search decoder.

- For RAG sequence we have harder problem.

    - **Thorough Decoding:** We have to run a separate beam search for each of your K documents (e.g., 10 beam searches) and this gives us a set of candidate answers. After this we take every candidate and re-score it using the full RAG-Sequence formula (summing its probability across all 10 documents). This approach accurate but very slow.

    - **Fast Decoding:** This is cheaper approximation and we run 10 beam searches. Assume that if an answer $y$ wasn't in the beam for document $z$, its probability $p(y|x,z)$ is just 0. This avoids the slow re-scoring step.

## Implementing the RAG

In [9]:
import faiss
import numpy as np
import torch.nn as nn
import seaborn as sns# for data visualization
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from torch.utils.data import Dataset, DataLoader


In [10]:
import torch
torch.manual_seed(42)
np.random.seed(42)

In [11]:
from transformers import (
    BertModel, BertTokenizer,
    BartForConditionalGeneration, BartTokenizer,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

### Configuring the RAG Model